In [1]:


#------------------------------------------------ Begin_Librairie ----------------------------------------

import pandas as pd 

from time import sleep

import datetime

from pandas import ExcelWriter

import string

import re

import pdfplumber

import os


from selenium import webdriver

from selenium.webdriver.common.by import By

from selenium.webdriver.support.ui import WebDriverWait

from selenium.webdriver.support import expected_conditions as EC


from selenium.webdriver.common.keys import Keys

from webdriver_manager.chrome import ChromeDriverManager

from selenium.webdriver.chrome.options import Options

from selenium.webdriver.common.alert import Alert
import requests
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
import html

import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import camelot


In [2]:
#------------------------------------------------ Begin_ fileName ----------------------------------------
regulatorName = 'FR ACP' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.1")
now=datetime.datetime.now()
filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":",".")[:-7])

scriptfolder = f"C:\\Users\\wuj1\\OneDrive - Moody's\\Desktop\\Regulator\\{regulatorName}"

#scriptfolder=os.path.dirname(os.path.abspath(__file__)) ## to decomment for the production environment
os.chdir(scriptfolder)
tempfolder=os.path.join(scriptfolder, 'tempfolder') #if files are downloaded during the process

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


Running FR ACP Web Scraping Tool v.1.1


In [3]:
# %%

#------------------------------------------------ Begin_chromedriver ----------------------------------------

#Starting Chrome driver, set to download files in tempfolder

chromeOptions = webdriver.ChromeOptions()

prefs = {"plugins.always_open_pdf_externally": True,

		 "download.prompt_for_download": False,

		 "download.default_directory" : tempfolder}

chromeOptions.add_experimental_option("prefs",prefs)

driver = webdriver.Chrome(options=chromeOptions)

driver.maximize_window()




# %%

In [4]:
# %%

#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


def find_zip_code(string):
    match = re.search(r'\d{5}', string)
    if match:
        return match.group()
    else:
        return ''

In [5]:
#------------------------------------------------ Begin_Variable ----------------------------------------


sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [], 
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [], 
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [], 
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [], 
          'Phone - Mother company': [], 'Check': []}

processdate = now.strftime('%Y-%m-%d')

BASE_URL = "https://api.regafi.banque-france.fr/regafi-en/v1/en/entities/exportall"
APP_ID = "257083457715b20ccf93f61712930b42"
MAX_WORKERS = 8
headers = {"accept": "application/json"}

regdict={

        #regulatorName + ' 1': 'https://api.regafi.banque-france.fr/regafi-en/v1/en/entities/exportall',
        regulatorName + ' 10': 'https://acpr.banque-france.fr/fr/professionnels/lacpr-vous-accompagne/banque/decouvrir-le-controle-bancaire/entites-systemiques-du-secteur-bancaire',
        regulatorName + ' 13': 'https://acpr.banque-france.fr/fr/professionnels/vos-outils-et-services/consulter-les-registres/registre-des-agents-financiers-et-des-organismes-dassurance',

        }



Typology={

       regulatorName + ' 1': "Companies authorised to carry on banking activities, electronic money, financial services or payment services under the monetary and financial regulation",
       regulatorName + ' 10': "Liste des Établissements d’importance systémique du secteur bancaire",
       regulatorName + ' 13': "Liste des organismes d'assurance actifs",


        }

In [6]:

#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ ")
    if reg == regulatorName + ' 1':
        r = requests.get(BASE_URL, headers=headers, params={"appId": APP_ID, "page": 1}, timeout=60)
        r.raise_for_status()
        first = json.loads(r.content.decode("utf-8-sig"))

        nb_pages = int(first.get("nbPages", 1))
        all_items = list(first.get("data", []))
        print(f"Total pages: {nb_pages}, items in first page: {len(all_items)}, remaining pages: {nb_pages - 1}")

        def fetch_page(p):
            # per-thread session
            session = requests.Session()
            retries = 5
            for attempt in range(retries):
                try:
                    r = session.get(BASE_URL, headers=headers, params={"appId": APP_ID, "page": p}, timeout=60)
                    if r.status_code in (500, 502, 503, 504):
                        raise requests.HTTPError(f"{r.status_code} server error")
                    r.raise_for_status()
                    return json.loads(r.content.decode("utf-8-sig")).get("data", [])
                except Exception as e:
                    if attempt == retries - 1:
                        raise
                    sleep_s = 2 ** attempt
                    time.sleep(sleep_s)

        try:
            with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
                futures = {ex.submit(fetch_page, p): p for p in range(2, nb_pages + 1)}
                for f in as_completed(futures):
                    page_items = f.result()
                    all_items.extend(page_items)
                    done_pages = 1 + (len(all_items) // max(1, len(first.get("data", []))))
                    print(f"Fetched page {futures[f]} with {len(page_items)} items, total items: {len(all_items)}, pages done: {done_pages}/{nb_pages}")
        except Exception as e:
            print("Error fetching pages:", e)

        print("pages:", nb_pages, "entities:", len(all_items))


        for index_ , item in enumerate(all_items):
            title_ = list(item.keys())
            d = item.get(title_[0], {})
            keys = list(d)
            first_key = keys[0] if len(keys) > 0 else None
            first_value = d.get(first_key) if first_key is not None else ''
            second_key = keys[1] if len(keys) > 1 else None
            second_value = d.get(second_key) if second_key is not None else ''
            print(first_key,  html.unescape(first_value))
            print(second_key, html.unescape(second_value))

            detail = item.get(title_[1], {})

            siren = detail.get("siren", "")
            num_lei = detail.get("num_lei", "")
            bank_code = detail.get("bank_code", "")
            status = detail.get("status", "")
            description = detail.get("description", "")
            auth_type = detail.get("auth_type", "")
            agrement_date = detail.get("agrement_date", "")
            num_uniqueid = detail.get("num_uniqueid", "")
            legal_form = detail.get("legal_form", "")
            address = detail.get("address", "")
            postal_code = detail.get("postal_code", "")
            city = detail.get("city", "")
            country = detail.get("country", "")

            print(f"Current index: {index_}, Total items: {len(all_items)}")

            sqldict['Name'].append(html.unescape(first_value))
            sqldict['InternalID_1'].append(html.unescape(second_value))
            sqldict['InternalID_1_type'].append(second_key)
            sqldict['Address_1'].append(address)
            sqldict['City'].append(city)
            sqldict['Cntry'].append(country)
            sqldict['LEI Code'].append(num_lei)
            sqldict['BIC SWIFT Code'].append(bank_code)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['License_Type'].append(auth_type)
            sqldict['Typology'].append(description)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            if siren:
                sqldict['InternalID_2'].append(siren)
                sqldict['InternalID_2_type'].append("National identifier (SIREN number for French entities)")
            if num_uniqueid:
                sqldict['InternalID_3'].append(num_uniqueid)
                sqldict['InternalID_3_type'].append("National identifier (unique ID for French entities)")
            sqldict = bourange_same_length_array(sqldict)
            # print(
            #     f"siren:{siren}\n"
            #     f"num_lei:{num_lei}\n"
            #     f"bank_code:{bank_code}\n"
            #     f"status:{status}\n"
            #     f"description:{description}\n"
            #     f"auth_type:{auth_type}\n"
            #     f"agrement_date:{agrement_date}\n"
            #     f"num_uniqueid:{num_uniqueid}\n"
            #     f"legal_form:{legal_form}\n"
            #     f"address:{address}\n"
            #     f"postal_code:{postal_code}\n"
            #     f"city:{city}\n"
            #     f"country:{country}"
            # )
            print("-" * 50)
    elif reg == regulatorName + ' 10':
        html = requests.get(regdict[reg], timeout=30).text
        soup = BeautifulSoup(html, "html.parser")
        h2 = soup.find("h2", id="Entits-systmiques-du-secteur-bancaire-79229")
        if not h2:
            raise SystemExit("h2 not found")
        div = h2.find_next_sibling("div")
        if not div:
            raise SystemExit("sibling div not found")

        table = div.find("table")
        if not table:
            raise SystemExit("table not found")

        target_row = None
        for tr in table.find_all("tr"):
            if "listes officielles d'entités et coussins associés en france" in tr.get_text(" ", strip=True).lower():
                target_row = tr
                break

        if not target_row:
            raise SystemExit("target row not found")

        tds = target_row.find_all("td")
        if len(tds) < 2:
            raise SystemExit("not enough tds in target row")

        first_link = tds[0].find_all("a")[-1] if tds[0].find_all("a") else None
        second_link = tds[1].find_all("a")[-1] if tds[1].find_all("a") else None

        if not first_link or not second_link:
            raise SystemExit("could not find last links in both tds")

        links = [urljoin(regdict[reg], first_link.get("href")), urljoin(regdict[reg], second_link.get("href"))]

        pdf_links = []
        for link in links:
            page_html = requests.get(link, timeout=30).text
            page_soup = BeautifulSoup(page_html, "html.parser")
            a = page_soup.find("a", attrs={"role": "button", "data-file-extension": "pdf"})
            h1 = page_soup.find("h1").text.strip() if page_soup.find("h1") else ""
            if not a or not a.get("href"):
                raise SystemExit(f"pdf button link not found in {link}")
            pdf_links.append(urljoin(link, a.get("href")))

        print("PDF links:")
        for p in pdf_links:
            print(p)
            sleep(2)
            driver.get(p)
            
            print("PDF Title:", h1)
                

            print(os.listdir(tempfolder))
            sleep(2)
            pdf_file = os.listdir(tempfolder)[0]
            filePath = os.path.join(tempfolder, pdf_file)

            tables = camelot.read_pdf(filePath, pages='all', flavor='lattice')       

            for k in range(tables.n):
                df_Table = tables[k].df
                
                for j in range(1,len(df_Table)):
                    name = ' '.join([item.strip() for item in df_Table[0][j].splitlines() if item !=''])
                    name = name.replace('*', ' ')
                    address = ' '.join([item.strip() for item in df_Table[1][j].splitlines() if item !=''])
                    #print(f"Name: {name}, Address: {address}, Zip Code: {find_zip_code(address)}")
                    sqldict['Name'].append(name)
                    sqldict['ListProcessDate'].append(processdate)
                    sqldict['Address_1'].append(address)
                    sqldict['Zip'].append(find_zip_code(address))
                    sqldict['RegCtry'].append(reg.split()[0])
                    sqldict['RegCode'].append(reg.split()[1])
                    sqldict['ListCode'].append(reg.split()[2])
                    sqldict['RegulationType'].append('Regulated')
                    sqldict['ListName'].append(Typology[reg])
                    sqldcit = bourange_same_length_array(sqldict)
            os.remove(filePath)
    elif reg == regulatorName + ' 13':
        html = requests.get(regdict[reg], timeout=30).text
        soup = BeautifulSoup(html, "html.parser")

        h3 = None
        for tag in soup.find_all("h3"):
            if "télécharger la liste des organismes d'assurance actifs".lower() in tag.get_text(" ", strip=True).lower():
                h3 = tag
                break

        if not h3:
            raise SystemExit("h3 not found")

        div = h3.find_next_sibling("div")
        if not div:
            raise SystemExit("sibling div not found")

        links = div.find_all("a", attrs={"role": "button", "data-file-extension": "xlsx"})
        if not links:
            raise SystemExit("no matching xlsx links found")

        last_link = links[-1].get("href")
        if not last_link:
            raise SystemExit("last link missing href")

        final_url = urljoin(regdict[reg], last_link)
        print(final_url)
        driver.get(final_url)
        print(os.listdir(tempfolder))
        sleep(2)
        pdf_file = os.listdir(tempfolder)[0]
        filePath = os.path.join(tempfolder, pdf_file)
        sleep(1)
        df_list_3 = pd.read_excel(filePath)
        for index, row in df_list_3.iterrows():
            id_ = row.iloc[0]
            name_ = row.iloc[1]
            topology = row.iloc[2]
            address_ = row.iloc[8]
            code_zip = row.iloc[9]
            city_ = row.iloc[10] 
            lei = row.iloc[12]
            sqldict['Name'].append(name_)
            sqldict['ListProcessDate'].append(processdate)
            sqldict['Address_1'].append(address_)
            sqldict['Zip'].append(code_zip)
            sqldict['City'].append(city_)
            sqldict['LEI Code'].append(lei)
            sqldict['Typology'].append(topology)
            sqldict['RegCtry'].append(reg.split()[0])
            sqldict['RegCode'].append(reg.split()[1])
            sqldict['ListCode'].append(reg.split()[2])
            sqldict['RegulationType'].append('Regulated')
            sqldict['ListName'].append(Typology[reg])
            sqldcit = bourange_same_length_array(sqldict)
        os.remove(filePath)
            

        


[INFO] : Working 1/2 _(FR ACP 10)_ 
PDF links:
https://acpr.banque-france.fr/system/files/2025-12/20251201_Liste_EISm_2025_au_titre_2024.pdf
PDF Title: Liste des Autres établissements d’importance systémique (A-EIS) au titre de l’exercice 2024 conformément aux dispositions de l'article L511-41-1 A VII du Code monétaire et financier
['cf4aa08c-9627-4b4c-95ee-c2560621363e.tmp']
https://acpr.banque-france.fr/system/files/2025-12/20251201_Liste_AEIS_2025_au_titre_2024.pdf
PDF Title: Liste des Autres établissements d’importance systémique (A-EIS) au titre de l’exercice 2024 conformément aux dispositions de l'article L511-41-1 A VII du Code monétaire et financier
['45879680-c56a-4016-b3d6-c8be6da7f1fb.tmp']
[INFO] : Working 2/2 _(FR ACP 13)_ 
https://acpr.banque-france.fr/system/files/2026-03/20260302_liste_des_organismes_d_assurances_actifs.xlsx
['b07a7e78-7ff1-4d19-903d-da396bfe52bc.tmp']


In [7]:
sqldict = bourange_same_length_array(sqldict)

In [ ]:
# %%

#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------

os.chdir(scriptfolder)

df=pd.DataFrame(sqldict)
df = df[df['Name']!='']
df.to_excel(filename, index=False)

driver.quit()

sleep(3)